# 🌍 Project 6: Agri-Carbon Credit & Soil Organic Carbon (SOC) dMRV Diagnostic
**Pipeline Stage:** Python Data Hygiene, IPCC Tier-2 SOC Modeling & Feature Engineering  
**Target Domain:** Carbon Credit Developers, ESG Funds & Regenerative Ag Aggregators

In [ ]:
import pandas as pd
import numpy as np

# 1. Ingest raw field sequestration log
df_raw = pd.read_csv('../data/field_carbon_sequestration_raw.csv')
print(f"Raw Field Records Ingested: {len(df_raw)}")
df_raw.head()

## 🧹 Data Hygiene & Text Standardization

In [ ]:
# Clean whitespace and standardize text columns
df_clean = df_raw.copy()
text_cols = ['fpo_name', 'state', 'district', 'practice_type']
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Impute missing risk flags based on delta_soc_pct thresholds
def impute_risk(row):
    if pd.isna(row['risk_flag']):
        if row['delta_soc_pct'] < 0.25:
            return 'High Risk (Low Accretion)'
        elif row['delta_soc_pct'] > 0.65:
            return 'Moderate Risk (High Volatility)'
        else:
            return 'Low Risk (Verified)'
    return row['risk_flag']

df_clean['risk_flag'] = df_clean.apply(impute_risk, axis=1)
print(f"Cleaned Records: {len(df_clean)}")

## 🧮 IPCC Tier-2 Soil Organic Carbon (SOC) Feature Engineering

In [ ]:
# Verify IPCC Tier-2 SOC Stock (Tonnes C/Ha) = SOC % * Bulk Density * Depth (cm) * 100
df_clean['calc_baseline_soc_stock'] = (df_clean['baseline_soc_pct'] / 100.0) * df_clean['bulk_density_g_cm3'] * df_clean['soil_depth_cm'] * 100.0
df_clean['calc_project_soc_stock'] = (df_clean['project_soc_pct'] / 100.0) * df_clean['bulk_density_g_cm3'] * df_clean['soil_depth_cm'] * 100.0
df_clean['calc_delta_soc_stock'] = df_clean['calc_project_soc_stock'] - df_clean['calc_baseline_soc_stock']

# Calculate Gross Sequestration tCO2e/Ha = delta_soc_stock * 3.6667 (44/12 molecular weight ratio)
df_clean['calc_gross_seq_ha'] = df_clean['calc_delta_soc_stock'] * 3.6667
df_clean['calc_net_seq_ha'] = (df_clean['calc_gross_seq_ha'] - df_clean['operational_emissions_tco2e_ha']).clip(lower=0.10)

# Total Net Sequestration tCO2e = net_seq_ha * field_area_ha
df_clean['calc_total_net_seq'] = df_clean['calc_net_seq_ha'] * df_clean['field_area_ha']

# 15% Permanence Buffer Reserve Deduction
df_clean['calc_buffer_deduction'] = df_clean['calc_total_net_seq'] * 0.15
df_clean['calc_issuable_credits'] = df_clean['calc_total_net_seq'] - df_clean['calc_buffer_deduction']

# Carbon Revenue Split ($25 / tCO2e)
df_clean['calc_total_revenue_usd'] = df_clean['calc_issuable_credits'] * 25.00
df_clean['calc_farmer_share_usd'] = df_clean['calc_total_revenue_usd'] * 0.70
df_clean['calc_fpo_share_usd'] = df_clean['calc_total_revenue_usd'] * 0.30

print("Summary Metrics:")
print(f"Total Hectares: {df_clean['field_area_ha'].sum():.2f} Ha")
print(f"Total Net Sequestration: {df_clean['calc_total_net_seq'].sum():.2f} tCO2e")
print(f"Total Issuable Credits: {df_clean['calc_issuable_credits'].sum():.2f} tCO2e")
print(f"Total Carbon Revenue: ${df_clean['calc_total_revenue_usd'].sum():,.2f}")
print(f"Farmer Payout Share (70%): ${df_clean['calc_farmer_share_usd'].sum():,.2f}")
print(f"FPO/Developer Share (30%): ${df_clean['calc_fpo_share_usd'].sum():,.2f}")

## 💾 Export Processed Summary Tables

In [ ]:
# Export clean CSV
df_clean.to_csv('../data/field_carbon_sequestration_clean.csv', index=False)

# Export cluster aggregated summary
cluster_summary = df_clean.groupby(['state', 'district', 'fpo_name']).agg(
    total_fields=('field_id', 'count'),
    total_area_ha=('field_area_ha', 'sum'),
    avg_delta_soc_pct=('delta_soc_pct', 'mean'),
    total_net_sequestration_tco2e=('calc_total_net_seq', 'sum'),
    total_issuable_credits=('calc_issuable_credits', 'sum'),
    total_revenue_usd=('calc_total_revenue_usd', 'sum'),
    farmer_share_usd=('calc_farmer_share_usd', 'sum'),
    fpo_share_usd=('calc_fpo_share_usd', 'sum')
).reset_index()

cluster_summary.to_csv('../data/soc_cluster_summary.csv', index=False)
print("Pipeline complete! Exported clean datasets.")